In [ ]:
from pathlib import Path
import sys
import time
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
import shap
from sklearn.metrics import roc_auc_score

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.split import make_time_folds, expanding_window_splits
from src.features.build_features import build_features

DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
REPORTS = PROJECT_ROOT / "reports"
REPORTS_FIGURES = PROJECT_ROOT / "reports" / "figures"

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 100

# Загружаем данные и FE
train = pd.read_parquet(DATA_INTERIM / "train.parquet", engine="pyarrow")
test = pd.read_parquet(DATA_INTERIM / "test.parquet", engine="pyarrow")
rename_map = {c: c.replace("id-", "id_") for c in test.columns if c.startswith("id-")}
test = test.rename(columns=rename_map)

folds = make_time_folds(train["TransactionDT"], n_splits=5)
target = train["isFraud"].astype(np.int8)

train_fe, test_fe = build_features(
    train=train.copy(),
    test=test.copy(),
    target=target,
    folds=folds,
    verbose=False,
)

# Выбранные фичи и best params
with open(REPORTS / "selected_features_v4.json") as f:
    features_to_keep = json.load(f)["features_to_keep"]

with open(REPORTS / "best_params_v5.json") as f:
    best_params = json.load(f)["params"]

y = train_fe["isFraud"].astype(np.int8)
drop_cols = ["TransactionID", "isFraud", "TransactionDT", "uid"]
X = train_fe.drop(columns=drop_cols)[features_to_keep]
cat_cols = X.select_dtypes(include=["category"]).columns.tolist()

# Добавляем фиксированные параметры к best_params
lgb_params = {
    "objective": "binary",
    "metric": "auc",
    "verbose": -1,
    "n_jobs": -1,
    "random_state": 42,
    **best_params,
}

print(f"X: {X.shape}")
print(f"Best params loaded")
print(f"SHAP version: {shap.__version__}")

In [ ]:
splits = expanding_window_splits(folds, n_splits=5)
train_idx, valid_idx = splits[-1]  # fold 4

X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
X_va, y_va = X.iloc[valid_idx], y.iloc[valid_idx]

t0 = time.time()
train_data = lgb.Dataset(X_tr, label=y_tr, categorical_feature=cat_cols)
valid_data = lgb.Dataset(X_va, label=y_va, categorical_feature=cat_cols,
                          reference=train_data)

model = lgb.train(
    params=lgb_params,
    train_set=train_data,
    num_boost_round=3000,
    valid_sets=[valid_data],
    callbacks=[
        lgb.early_stopping(stopping_rounds=150, verbose=False),
        lgb.log_evaluation(period=0),
    ],
)

preds_va = model.predict(X_va, num_iteration=model.best_iteration)
auc = roc_auc_score(y_va, preds_va)
print(f"Model trained in {time.time() - t0:.0f}s")
print(f"Fold 4 AUC: {auc:.5f}  (should match v5 fold 4: 0.94780)")
print(f"Best iteration: {model.best_iteration}")

In [ ]:
# Стратифицированный sample на 5000 строк из valid
rng = np.random.RandomState(42)

# Берём 5000 из valid с сохранением пропорции fraud/не-fraud
valid_pos_idx = valid_idx[y.iloc[valid_idx].values == 1]
valid_neg_idx = valid_idx[y.iloc[valid_idx].values == 0]

n_pos = 500  # 10% примерно от 5000, хотя в данных ~3.5%
n_neg = 4500

sample_pos = rng.choice(valid_pos_idx, size=n_pos, replace=False)
sample_neg = rng.choice(valid_neg_idx, size=n_neg, replace=False)
sample_idx = np.concatenate([sample_pos, sample_neg])

X_sample = X.iloc[sample_idx].reset_index(drop=True)
y_sample = y.iloc[sample_idx].reset_index(drop=True)

print(f"Sample size: {len(X_sample)}")
print(f"Fraud in sample: {y_sample.mean() * 100:.1f}%")
print()

# TreeExplainer — быстрый для LightGBM
t0 = time.time()
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_sample)
print(f"SHAP values computed in {time.time() - t0:.0f}s")
print(f"Shape: {shap_values.shape if hasattr(shap_values, 'shape') else type(shap_values)}")

In [ ]:
# Beeswarm plot
fig = plt.figure(figsize=(10, 10))
shap.summary_plot(
    shap_values,
    X_sample,
    max_display=20,
    show=False,
    plot_size=(10, 10),
)
plt.title("SHAP Summary Plot — top 20 features\n"
          "Каждая точка — транзакция; цвет = значение фичи; "
          "положение по X = вклад в предсказание (SHAP value)",
          fontsize=11, pad=20)
plt.tight_layout()
plt.savefig(REPORTS_FIGURES / "08_shap_summary.png", bbox_inches="tight", dpi=150)
plt.show()

In [ ]:
fig = plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values,
    X_sample,
    plot_type="bar",
    max_display=20,
    show=False,
    plot_size=(10, 8),
)
plt.title("SHAP Feature Importance — top 20 by mean |SHAP|",
          fontsize=11, pad=20)
plt.tight_layout()
plt.savefig(REPORTS_FIGURES / "09_shap_bar.png", bbox_inches="tight", dpi=150)
plt.show()

In [ ]:
# Dependence plot для uid_te — явно указываем числовую interaction
fig, ax = plt.subplots(figsize=(11, 6))
shap.dependence_plot(
    "uid_te",
    shap_values,
    X_sample,
    interaction_index="TransactionAmt",  # явно, без auto
    ax=ax,
    show=False,
)
plt.title("SHAP Dependence Plot: uid_te\n"
          "Как значение uid_te влияет на SHAP value; цвет = TransactionAmt",
          fontsize=11, pad=15)
plt.tight_layout()
plt.savefig(REPORTS_FIGURES / "10_shap_dependence_uid_te.png",
            bbox_inches="tight", dpi=150)
plt.show()

In [ ]:
# Вычисляем предсказания для sample
sample_preds = model.predict(X_sample, num_iteration=model.best_iteration)

# Найдём интересные примеры
# TP: high prob + actually fraud
mask_tp = (sample_preds > 0.9) & (y_sample.values == 1)
# FP: high prob but actually not fraud
mask_fp = (sample_preds > 0.9) & (y_sample.values == 0)

print(f"TP candidates (prob > 0.9): {mask_tp.sum()}")
print(f"FP candidates (prob > 0.9): {mask_fp.sum()}")

# Выбираем по одному из каждого
tp_idx = np.where(mask_tp)[0][0]  # первый TP
fp_idx = np.where(mask_fp)[0][0]  # первый FP

print()
print(f"TP example: idx={tp_idx}, prob={sample_preds[tp_idx]:.4f}, "
      f"true={y_sample.iloc[tp_idx]}")
print(f"FP example: idx={fp_idx}, prob={sample_preds[fp_idx]:.4f}, "
      f"true={y_sample.iloc[fp_idx]}")

In [ ]:
# SHAP values для конкретного примера
# Используем waterfall — современный вариант force plot, который сохраняется в PNG

import shap

# TP example
shap_values_tp = shap_values[tp_idx]
expected_value = explainer.expected_value
if hasattr(expected_value, "__len__"):
    expected_value = expected_value[1] if len(expected_value) > 1 else expected_value[0]

fig = plt.figure(figsize=(10, 8))

shap.waterfall_plot(
    shap.Explanation(
        values=shap_values_tp,
        base_values=expected_value,
        data=X_sample.iloc[tp_idx].values,
        feature_names=X_sample.columns.tolist(),
    ),
    max_display=15,
    show=False,
)
plt.title(f"TP: fraud transaction, model predicted {sample_preds[tp_idx]:.3f}\n"
          f"Top features pushing toward fraud",
          fontsize=11, pad=15)
plt.tight_layout()
plt.savefig(REPORTS_FIGURES / "11_shap_force_tp.png",
            bbox_inches="tight", dpi=150)
plt.show()

In [ ]:
    # Force plot для FP (тот, где модель ошиблась)
fig = plt.figure(figsize=(10, 8))
shap_values_fp = shap_values[fp_idx]
shap.waterfall_plot(
    shap.Explanation(
        values=shap_values_fp,
        base_values=expected_value,
        data=X_sample.iloc[fp_idx].values,
        feature_names=X_sample.columns.tolist(),
    ),
    max_display=15,
    show=False,
)
plt.title(f"FP: NOT fraud, but model predicted {sample_preds[fp_idx]:.3f}\n"
          f"Where the model is overconfident — error case analysis",
          fontsize=11, pad=15)
plt.tight_layout()
plt.savefig(REPORTS_FIGURES / "12_shap_force_fp.png",
            bbox_inches="tight", dpi=150)
plt.show()

print("SHAP analysis complete. 5 figures saved to reports/figures/")